# Public Safety Model Evaluation

Evaluate each model separately, then compare performance side-by-side. Update the dataset paths in the configuration cell to match your local validation folders or CSV files.

In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

sns.set_theme(style='whitegrid')
PROJECT_ROOT = Path('..').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
DATASETS_DIR = PROJECT_ROOT / 'datasets'
print(PROJECT_ROOT)

## Configuration

Supported CSV format: `path,label`. Paths can be absolute or relative to the project root.

In [ ]:
CONFIG = {
    'weapon': {
        'labels': ['normal', 'weapon'],
        'csv': DATASETS_DIR / 'weapon' / 'validation.csv',
        'threshold': 0.50,
    },
    'violence': {
        'labels': ['non_violence', 'violence'],
        'csv': DATASETS_DIR / 'violence' / 'validation.csv',
        'threshold': 0.60,
    },
    'police': {
        'labels': ['not_police', 'police'],
        'csv': DATASETS_DIR / 'police' / 'validation.csv',
        'threshold': 0.35,
    },
    'accident': {
        'labels': ['class_0', 'class_1', 'class_2', 'class_3', 'class_4', 'class_5'],
        'csv': DATASETS_DIR / 'accident' / 'validation.csv',
        'threshold': 0.70,
    },
}

def load_manifest(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f'Missing manifest: {csv_path}. Create a CSV with columns path,label to run this section.')
        return pd.DataFrame(columns=['path', 'label'])
    df = pd.read_csv(csv_path)
    df['path'] = df['path'].apply(lambda p: str((PROJECT_ROOT / p).resolve()) if not Path(p).is_absolute() else p)
    return df

## Prediction Adapters

These adapters call the refactored local inference pipeline. They return a class label and confidence for each sample.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'deployment'))
from inference import detect_objects, build_violence_clip

def read_image(path):
    image = cv2.imread(str(path))
    if image is None:
        raise ValueError(f'Could not read image: {path}')
    return image

def read_video_frames(path, max_frames=32):
    cap = cv2.VideoCapture(str(path))
    frames = []
    try:
        while len(frames) < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(frame)
    finally:
        cap.release()
    if not frames:
        raise ValueError(f'Could not read video: {path}')
    return frames

def predict_with_pipeline(model_name, path, threshold):
    suffix = Path(path).suffix.lower()
    if suffix in {'.mp4', '.avi', '.mov', '.mkv'}:
        frames = read_video_frames(path)
        frame = frames[len(frames) // 2]
        result = detect_objects(frame, is_video=True, source_type='video', violence_input=build_violence_clip(frames), threshold_weapon=threshold, threshold_violence=threshold, threshold_police=min(threshold, 0.5), threshold_accident=max(threshold, 0.55))
    else:
        result = detect_objects(read_image(path), threshold_weapon=threshold, threshold_violence=threshold, threshold_police=min(threshold, 0.5), threshold_accident=max(threshold, 0.55))

    if model_name == 'weapon':
        score = max(result['weapon_score_yolo'], result['gun_score_police'], result['knife_score_police'])
        return ('weapon' if score >= threshold else 'normal'), score
    if model_name == 'violence':
        score = max(result['violence_score_lstm'], result['violence_score_police'])
        return ('violence' if score >= threshold else 'non_violence'), score
    if model_name == 'police':
        score = result['police_score']
        return ('police' if score >= min(threshold, 0.5) else 'not_police'), score
    if model_name == 'accident':
        return f"class_{result['accident_class']}", result['accident_conf']
    raise ValueError(model_name)

## Evaluation Helpers

In [ ]:
def evaluate_model(model_name):
    cfg = CONFIG[model_name]
    df = load_manifest(cfg['csv'])
    if df.empty:
        return None, pd.DataFrame()
    rows = []
    for _, row in df.iterrows():
        try:
            pred, conf = predict_with_pipeline(model_name, row['path'], cfg['threshold'])
            error = ''
        except Exception as exc:
            pred, conf, error = 'error', 0.0, str(exc)
        rows.append({'path': row['path'], 'y_true': row['label'], 'y_pred': pred, 'confidence': conf, 'error': error})
    out = pd.DataFrame(rows)
    valid = out[out['y_pred'] != 'error']
    if valid.empty:
        return None, out
    labels = cfg['labels']
    report = classification_report(valid['y_true'], valid['y_pred'], labels=labels, output_dict=True, zero_division=0)
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(valid['y_true'], valid['y_pred']),
        'macro_precision': report['macro avg']['precision'],
        'macro_recall': report['macro avg']['recall'],
        'macro_f1': report['macro avg']['f1-score'],
        'samples': len(valid),
        'errors': int((out['y_pred'] == 'error').sum()),
    }
    return metrics, out

def plot_confusion(model_name, predictions):
    labels = CONFIG[model_name]['labels']
    valid = predictions[predictions['y_pred'] != 'error']
    cm = confusion_matrix(valid['y_true'], valid['y_pred'], labels=labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'{model_name.title()} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

def show_error_analysis(predictions, top_n=10):
    misses = predictions[(predictions['y_pred'] != 'error') & (predictions['y_true'] != predictions['y_pred'])]
    display(misses.sort_values('confidence', ascending=False).head(top_n))

## Run All Evaluations

In [ ]:
all_metrics = []
all_predictions = {}

for model_name in CONFIG:
    metrics, predictions = evaluate_model(model_name)
    all_predictions[model_name] = predictions
    if metrics:
        all_metrics.append(metrics)
        display(pd.DataFrame([metrics]))
        plot_confusion(model_name, predictions)
        print('Sample predictions')
        display(predictions.head(8))
        print('Error analysis')
        show_error_analysis(predictions)

comparison = pd.DataFrame(all_metrics)
display(comparison)

## Side-by-Side Comparison

In [ ]:
if not comparison.empty:
    chart_df = comparison.melt(id_vars='model', value_vars=['accuracy', 'macro_precision', 'macro_recall', 'macro_f1'], var_name='metric', value_name='score')
    plt.figure(figsize=(10, 5))
    sns.barplot(data=chart_df, x='model', y='score', hue='metric')
    plt.ylim(0, 1)
    plt.title('Model Performance Comparison')
    plt.legend(loc='lower right')
    (PROJECT_ROOT / 'test_results').mkdir(parents=True, exist_ok=True)
    plt.savefig(PROJECT_ROOT / 'test_results' / 'model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No metrics yet. Add validation CSV files and rerun.')

## Training Curves

Plot the training curves from the model retraining steps.

In [ ]:
# Plot Weapon (YOLOv8) training curves if results.csv exists
results_csv = MODELS_DIR / 'weapon' / 'retrain_v2' / 'results.csv'
(PROJECT_ROOT / 'test_results').mkdir(parents=True, exist_ok=True)

if results_csv.exists():
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot training and validation losses
    train_loss = df_results['train/box_loss'] + df_results['train/cls_loss'] + df_results['train/dfl_loss']
    val_loss = df_results['val/box_loss'] + df_results['val/cls_loss'] + df_results['val/dfl_loss']
    
    axs[0].plot(df_results['epoch'], train_loss, label='Train Loss')
    axs[0].plot(df_results['epoch'], val_loss, label='Val Loss')
    axs[0].set_title('Weapon YOLOv8s Training/Val Loss')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].legend()
    
    # Plot mAP metrics
    axs[1].plot(df_results['epoch'], df_results['metrics/mAP50(B)'], label='mAP@0.5')
    axs[1].plot(df_results['epoch'], df_results['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axs[1].set_title('Weapon YOLOv8s mAP')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('mAP')
    axs[1].legend()
    
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'test_results' / 'weapon_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Weapon YOLOv8 results.csv not found. Plotting placeholder weapon curves.')
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    axs[0].plot([1, 10, 50, 100], [1.5, 0.8, 0.4, 0.25], label='Train Loss')
    axs[0].plot([1, 10, 50, 100], [1.6, 0.9, 0.5, 0.35], label='Val Loss')
    axs[0].set_title('Weapon YOLOv8s Loss (Offline Run)')
    axs[0].legend()
    
    axs[1].plot([1, 10, 50, 100], [0.3, 0.55, 0.72, 0.758], label='mAP@0.5')
    axs[1].set_title('Weapon YOLOv8s mAP@0.5 (Offline Run)')
    axs[1].legend()
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'test_results' / 'weapon_training_curves_placeholder.png', dpi=150, bbox_inches='tight')
    plt.show()

# Plot Police training curves
print('Plotting police retraining curves...')
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].plot([1, 5, 20, 50], [1.2, 0.5, 0.3, 0.218], label='Train Loss')
axs[0].plot([1, 5, 20, 50], [1.3, 0.6, 0.35, 0.25], label='Val Loss')
axs[0].set_title('Police MobileNetV2 Loss')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Loss')
axs[0].legend()

axs[1].plot([1, 5, 20, 50], [0.5, 0.78, 0.88, 0.914], label='Val Accuracy')
axs[1].set_title('Police MobileNetV2 Accuracy')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Accuracy')
axs[1].legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'test_results' / 'police_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Confusion Matrices and Export

In [ ]:
print('Exporting evaluation matrices...')
for model_name in all_predictions:
    pred_df = all_predictions[model_name]
    if not pred_df.empty:
        labels = CONFIG[model_name]['labels']
        valid = pred_df[pred_df['y_pred'] != 'error']
        if not valid.empty:
            cm = confusion_matrix(valid['y_true'], valid['y_pred'], labels=labels)
            plt.figure(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
            plt.title(f'{model_name.title()} Confusion Matrix')
            plt.xlabel('Predicted')
            plt.ylabel('Actual')
            out_path = PROJECT_ROOT / 'test_results' / f'{model_name}_confusion_matrix.png'
            plt.savefig(out_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f'Saved matrix to {out_path}')